# 01 — Évaluation du retrieval (ArchéoGuide)

Notebook typique d'évaluation RAG : comparer **vector**, **BM25** et **hybrid** sur le jeu de référence.

**Métriques** : Hit Rate@k, MRR, latence (moyenne / P95).

**Prérequis** : Qdrant démarré, collection indexée, `OPENAI_API_KEY` dans `.env`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from eval.performance_study import evaluate_retrieval_with_latency
from eval.retrieval_eval import load_ground_truth, run_evaluation
from rag.config import get_settings

settings = get_settings()
queries = load_ground_truth()
TOP_K = 5
print(f"{len(queries)} questions | collection={settings.qdrant_collection} | top_k={TOP_K}")

## Comparaison des modes

In [ ]:
rows = []
for mode in ("vector", "bm25", "hybrid"):
    print(f"Évaluation mode={mode}…")
    result = evaluate_retrieval_with_latency(queries, mode, TOP_K, settings)
    rows.append({
        "mode": result["mode"],
        "hit_rate": result["metrics"]["hit_rate"],
        "mrr": result["metrics"]["mrr"],
        "mean_ms": result["latency"]["mean_ms"],
        "p50_ms": result["latency"]["p50_ms"],
        "p95_ms": result["latency"]["p95_ms"],
    })

df_retrieval = pd.DataFrame(rows)
df_retrieval

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df_retrieval.plot(x="mode", y=["hit_rate", "mrr"], kind="bar", ax=axes[0], rot=0)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Qualité retrieval")
axes[0].set_ylabel("Score")

df_retrieval.plot(
    x="mode", y=["mean_ms", "p95_ms"], kind="bar", ax=axes[1], rot=0,
    color=["#4C78A8", "#F58518"],
)
axes[1].set_title("Latence (ms)")
axes[1].set_ylabel("ms")

plt.tight_layout()
plt.show()

best = df_retrieval.sort_values(["hit_rate", "mrr"], ascending=False).iloc[0]
print(
    f"Meilleur mode : {best['mode']} "
    f"(hit={best['hit_rate']:.0%}, mrr={best['mrr']:.3f})"
)

## Export JSON (script officiel)

In [ ]:
report = run_evaluation(top_k=TOP_K)
print("best_mode =", report["best_mode"])
print("écrit dans eval/results/retrieval_eval_latest.json")